# Projekt: Datenqualität im E-Commerce
## Datensatz: Olist Brazilian E-Commerce (Kaggle)
**Autor:** Julio Nzonene  
**Datum:** April 2026  
**Tool:** DuckDB + Python

### Was wird untersucht?
Rohdaten aus einem brasilianischen E-Commerce-System werden auf Vollstaendigkeit, Konsistenz und logische Fehler geprueft, bevor irgendein Dashboard gebaut wird.

### Kernfragen:
- Welche Bestellungen haben fehlende Daten?
- Gibt es logisch unmoeglich Zeitstempel?
- Welche Muster deuten auf Systemfehler hin?

In [4]:
# Bibliotheken laden und Datenbankverbindung herstellen
import duckdb

con = duckdb.connect()

# Datensatz einladen
con.execute("""
    CREATE VIEW orders AS 
    SELECT * FROM read_csv_auto('C:/Users/julio/OneDrive/Portfolio/archive (1)/olist_orders_dataset.csv')
""")

print("Verbindung erfolgreich!")

Verbindung erfolgreich!


## Analyse 1: Fehlende Werte
Wir prüfen wie viele Bestellungen kritische Lücken haben.
Fehlende Lieferdaten bedeuten: Der Kunde weiss nicht ob sein Paket ankommt.

In [5]:
# Fehlende Werte in kritischen Spalten zählen
con.execute("""
    SELECT 
        COUNT(*) AS gesamt,
        COUNT(*) FILTER (WHERE order_approved_at IS NULL) AS keine_genehmigung,
        COUNT(*) FILTER (WHERE order_delivered_carrier_date IS NULL) AS kein_versanddatum,
        COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS keine_lieferung
    FROM orders
""").fetchdf()

,gesamt,keine_genehmigung,kein_versanddatum,keine_lieferung
0,99441,160,1783,2965


## Analyse 2: Logisch unmögliche Lieferungen
Eine Lieferung kann nicht vor dem Versand ankommen.
Solche Einträge deuten auf Fehler bei der Dateneingabe hin.

In [6]:
# Lieferungen die vor dem Versand ankamen
con.execute("""
    SELECT COUNT(*) AS unmögliche_lieferungen
    FROM orders
    WHERE order_delivered_customer_date < order_delivered_carrier_date
""").fetchdf()

,unmögliche_lieferungen
0,23


## Analyse 3: Lieferzeiten
Wir berechnen durchschnittliche, maximale und minimale Lieferzeiten.
Extreme Ausreißer deuten auf Datenfehler oder Systemprobleme hin.

In [7]:
# Lieferzeiten in Tagen berechnen
con.execute("""
    SELECT 
        ROUND(AVG(DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date)), 1) AS avg_liefertage,
        MAX(DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date)) AS max_liefertage,
        MIN(DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date)) AS min_liefertage
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
""").fetchdf()

,avg_liefertage,max_liefertage,min_liefertage
0,12.5,210,0


## Analyse 4: Extreme Ausreißer
Bestellungen mit über 100 Tagen Lieferzeit werden genauer untersucht.

In [8]:

# Top 10 längste Lieferzeiten
con.execute("""
    SELECT 
        order_id,
        order_status,
        DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date) AS liefertage,
        order_purchase_timestamp,
        order_delivered_customer_date
    FROM orders
    WHERE DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date) > 100
    ORDER BY liefertage DESC
    LIMIT 10
""").fetchdf()

,order_id,order_status,liefertage,order_purchase_timestamp,order_delivered_customer_date
0,ca07593549f1816d26a572e06dc1eab6,delivered,210,2017-02-21 23:31:27,2017-09-19 14:36:39
1,1b3190b2dfa9d789e1f14c05b647a14a,delivered,208,2018-02-23 14:57:35,2018-09-19 23:24:07
2,440d0d17af552815d15a9e41abe49359,delivered,196,2017-03-07 23:59:51,2017-09-19 15:12:50
3,285ab9426d6982034523a855f55a885e,delivered,195,2017-03-08 22:47:40,2017-09-19 14:00:04
4,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,195,2017-03-08 18:09:02,2017-09-19 14:33:17
5,0f4519c5f1c541ddec9f21b3bddd533a,delivered,194,2017-03-09 13:26:57,2017-09-19 14:38:21
6,47b40429ed8cce3aee9199792275433f,delivered,191,2018-01-03 09:44:01,2018-07-13 20:51:31
7,2fe324febf907e3ea3f2aa9650869fa5,delivered,190,2017-03-13 20:17:10,2017-09-19 17:00:07
8,c27815f7e3dd0b926b58552628481575,delivered,188,2017-03-15 23:23:17,2017-09-19 17:14:25
9,2d7561026d542c8dbd8f0daeadf67a43,delivered,188,2017-03-15 11:24:27,2017-09-19 14:38:18


## Analyse 5: Verdächtiges Muster - 19. September 2017
Fast alle Extremfälle wurden am selben Tag geliefert.
Das deutet auf einen Systemfehler bei der Dateneingabe hin - nicht auf echte Lieferungen.

In [10]:
# Wie viele Bestellungen wurden am 19.09.2017 geliefert?
con.execute("""
    SELECT 
        DATE(order_delivered_customer_date) AS lieferdatum,
        COUNT(*) AS anzahl
    FROM orders
    WHERE DATE(order_delivered_customer_date) = '2017-09-19'
    GROUP BY lieferdatum
""").fetchdf()

,lieferdatum,anzahl
0,2017-09-19,282


## Fazit & Handlungsempfehlung

| Befund | Zahl |
|---|---|
| Bestellungen ohne Lieferdatum | 2.965 |
| Logisch unmögliche Lieferungen | 23 |
| Verdächtiger Massenliefertag | 282 am 19.09.2017 |
| Durchschnittliche Lieferzeit | 12,5 Tage |
| Maximale Lieferzeit | 210 Tage |

**Kernaussage:** Bevor ein Dashboard oder ein KI-Modell auf diesen Daten aufgebaut wird, 
müssen mindestens 3 kritische Datenqualitätsprobleme behoben werden.
Dashboards auf schlechten Daten führen zu falschen Entscheidungen - nicht zu besseren.